# **Project 3 - Sales Analysis - ETL**

## Objectives

- Extract data from provided CSV file
- Clean it 
- Apply feature engineering if necessary 
- Remove any unnecessary columns
- Rename columns if necessary
- Save to a new CSV file


## Inputs

CSV files provided:

Sales_InvoiceData.csv


Note: original file stored in Data/OriginalFiles


## Outputs

CSV file created from ETL etc stored in Data/CleanedDataSets:

Sales_InvoiceData_Cleaned.csv



## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"

Used AI tool used to help with the ETL process:
- GitHub 
- Copilot 

See Documents/What_AI_Used_For.md for more details.


## Initalise Working Environment

In [22]:
#import libraries
import os
import numpy as np
import pandas as pd

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [23]:
#DataFrame vars for ETL
dfSales_DataSet = None
dfSales_DataSet_Work = None
dfTemp = None
dfCleaned = None

#missing values check vars
intLessThanZero = 0

#stores current directory
strCurrentDir = ""

#other vars
dictDataFrames = dict()
lstColumns = list()
intNum = 0

## Set Current Directory To Base Project Directory

In [24]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project3-SalesAnalysis


## Create Project Folder Structure

In [25]:
#create project folder structure
modETL.funcCreateDirectories()

Folder Structure Created Successfully!


# Section 1 - Extraction

- Read csv file sales data-set.csv as other two do not require ETL (boo!)
- Move into working files directory
- Read csv file into a pandas dataframe
- Get schema info -> column and row numbers
- Get first 5 records
- Get list of column datatypes
- Get detailed schema infomation

## Read csv File Into Variable For Processing

In [26]:
#read csv file into DataFrame
dfSales_DataSet = modETL.funcReadFileReturnDataFrame("Sales_InvoiceData.csv")
#save as working csv file
modETL.funcSaveDataFrameToWorkingFile(dfSales_DataSet)

#read file from working files folder
#ETL library returns a dictionary of all files in the folder with the attribute name
#set to the actual csv filename
dictDataFrames = modETL.funcReadWorkingFilesReturnDictionary()
dfSales_DataSet = dictDataFrames["Sales_InvoiceData_Working.csv"]

Folder Structure Created Successfully!
Read Sales_InvoiceData.csv Into DataFrame

Saved: Sales_InvoiceData To Working Folder
1 csv Files Read Into DataFrames

DataFrames Created:
Sales_InvoiceData_Working.csv




## Get Schema Info - Sales_Features_DataSet

In [27]:
modETL.funcGetStructure(dfSales_DataSet)
dfSales_DataSet.head()

DataFrame Structure:
<class 'pandas.DataFrame'>
RangeIndex: 189151 entries, 0 to 189150
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Unnamed: 0           189151 non-null  int64  
 1   INV_InvoiceID        189151 non-null  str    
 2   INV_InvoiceDate      189151 non-null  str    
 3   INV_SalesOrderID     189151 non-null  str    
 4   INV_SOLineNbr        189151 non-null  int64  
 5   INV_ItemID           189151 non-null  str    
 6   IMA_ItemName         189151 non-null  str    
 7   INV_ProdFam          188744 non-null  str    
 8   INV_InvoiceQty       189151 non-null  float64
 9   INV_InvoiceAmt       189151 non-null  float64
 10  INV_CustomerID       189151 non-null  str    
 11  CUS_BillName         189151 non-null  str    
 12  INV_TerritoryCodes   188905 non-null  str    
 13  CUS_BillContactName  107648 non-null  str    
 14  CUS_ShipMethod       189151 non-null  str    
dtypes: floa

,Unnamed: 0,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod
0,0,180851,01/03/2011,140079,3,17-972,Metalwork-Galvanised Retaining Brackets For Ca...,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
1,1,180851,01/03/2011,140079,4,17-625,M4 x 40mm Steel Pozi Pan Head M/Screw BZP Plated,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
2,2,180851,01/03/2011,140079,2,17-795/WHT,Metalwork White Recessing Frame Bewdley,Accessories,1.0,6.21,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
3,3,180834,01/03/2011,141347,1,SAV8/24/D/A ORB,8W 24V Ac/Dc Savona Edge Lit E/Sign Head Brass...,Savona,1.0,76.14,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
4,4,180834,01/03/2011,141347,2,l1919el-a,Legend Printed L19 Both Sides Suitable For All...,Luminaire Component,1.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day


## Observations - Sales_DataSet
189,151 rows
15 columns

Columns:

Unnamed,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod

Assuming Unnamed is an index column and can be dropped if necessary  

We can also see the data types:  

Unnamed : int64. 
INV_InvoiceID : int64  
INV_InvoiceDate : str  
INV_SalesOrderID : str  
INV_SOLineNbr : int64  
INV_ItemID : str   
IMA_ItemName : str   
INV_ProdFam : str   
INV_InvoiceQty : float64     
INV_InvoiceAmt : float64   
INV_CustomerID : str    
CUS_BillName : str    
INV_TerritoryCodes : str  
CUS_BillContactName : str  
CUS_ShipMethod : str  

In the Summary of Dataframe Structure for numerical values we see this:

DataFrame Structure:  
===================  
<class 'pandas.DataFrame'>  
RangeIndex: 189151 entries, 0 to 189150  
Data columns (total 15 columns):  

|     |Column               |Non-Null Count |  Dtype  |    
|-----|---------------------|---------------|---------|  
| 0   |Unnamed: 0           |189151 non-null|  int64  |  
| 1   |INV_InvoiceID        |189151 non-null|  str    |  
| 2   |INV_InvoiceDate      |189151 non-null|  str    |  
| 3   |INV_SalesOrderID     |189151 non-null|  str    |  
| 4   |INV_SOLineNbr        |189151 non-null|  int64  |  
| 5   |INV_ItemID           |189151 non-null|  str    |         
| 6   |IMA_ItemName         |189151 non-null|  str    |
| 7   |INV_ProdFam          |188744 non-null|  str    |
| 8   |INV_InvoiceQty       |189151 non-null|  float64|  
| 9   |INV_InvoiceAmt       |189151 non-null|  float64|  
| 10  |INV_CustomerID       |189151 non-null|  str    |
| 11  |CUS_BillName         |189151 non-null|  str    |
| 12  |INV_TerritoryCodes   |188905 non-null|  str    |
| 13  |CUS_BillContactName  |107648 non-null|  str    |
| 14  |CUS_ShipMethod       |189151 non-null|  str    |

dtypes: float64(2), int64(2), str(12)  
memory usage: 50.0 MB. 
None

Looks like there are some null values in the INV_ProdFam, INV_TerritoryCodes and CUS_BillContactName  

Summary of DataFrame Structure:
===============================
Numeric Columns:

|       |    Unnamed: 0 |INV_SOLineNbr   |INV_InvoiceQty|INV_InvoiceAmt| 
|-------|---------------|----------------|--------------|--------------|  
|count  |189151.000000  |189151.000000   |189151.000000 |189151.000000 |
|mean   | 94575.000000  |     3.500283   |    21.540247 |   296.874155 |
|std    | 54603.334721  |     4.690949   |   129.116010 |  1591.709197 |
|min    |     0.000000  |     1.000000   |     1.000000 | -1723.440000 |
|25%    | 47287.500000  |     1.000000   |     1.000000 |.   12.500000 |
|50%    | 94575.000000  |     2.000000   |     3.000000 |    47.320000 |
|75%    |141862.500000  |     4.000000   |    10.000000 |   161.210000 |
|max    |189150.000000  |   104.000000   | 10000.000000 |124635.000000 |


String Columns:

|       |INV_InvoiceID  |   INV_InvoiceDate |INV_SalesOrderID |INV_ItemID|  
|-------|---------------|-------------------|-----------------|----------|
|count  |189151         |            189151 |          189151 |    189151|   
|unique | 78001         |              1691 |           66793 |      7558|   
|top    |271275         |28/4/2011 00:00:00 |        Misc Inv |  CARRIAGE|   
|freq   |    61.        |               515 |             407 |     14752|     
  
<br>

|       |       IMA_ItemName|   INV_ProdFam| INV_CustomerID|  
|-------|-------------------|--------------|---------------|
|count  |             189151|        188744|         189151|   
|unique |               5862|           130|           1928|   
|top    | UK Carrier Service|  Fire Product|           S654|   
|freq   |              15674|         57593|           7145|     

<br>    

|        |    CUS_BillName| INV_TerritoryCodes| CUS_BillContactName| CUS_ShipMethod|  
|--------|----------------|-------------------|--------------------|---------------|
|count   |          189151|             188905|              107648|         189151|  
|unique  |            1597|                 34|                 777|             12| 
|top     |Rexel UK Limited|        South Coast|      Andrew Johnson|     UK Carrier|  
|freq    |            8034|              21566|                4824|         129502|  
  






**What Does This Mean?**

- Count is how many values are in the columns
- Mean is the average value of the column
- Std is the standard deviation of the column 
  Standard deviation is how far from the average (mean) the values are. The closer to 0 the
  more consistent the values are i.e. not much deviation from the mean. The deviation range 
  can be referred to as the Sigma. The measurement is based on the Alpha this is a percentage
  of the mean used to determine variance. ypically this is set to 5%
- Min is the minimum value of the column
- 25% is the value at the 25th percentile of the column
- 50% is the value between the 25th and 75th percentiles of the column
- 75% is the value at the 75th percentile of the column

The middle 50% is also called the Interquartile Range (IQR) and its width can be set by changing the
alpha value which is normally 5%

**What?**

Percentile is a technical term for a 25% segment of the data. 
50% is the mean, 25% is below that and 75% is 25%above that, this crude picture illustrates this:

 1st Percentile      2nd Percentile        3rd Percentile
0      -      25%  25%      -     75%   75%      -     100%  

Note: 2nd percentile can also be called Central Tendency

- Max is the maximum value of the column

What we can see is some columns need to be populated more as null seem to be present, let us look further..



# Get Column Data Types

In [28]:
#get column data types
print (f"Data Types: \n{dfSales_DataSet.dtypes}")

Data Types: 
Unnamed: 0               int64
INV_InvoiceID              str
INV_InvoiceDate            str
INV_SalesOrderID           str
INV_SOLineNbr            int64
INV_ItemID                 str
IMA_ItemName               str
INV_ProdFam                str
INV_InvoiceQty         float64
INV_InvoiceAmt         float64
INV_CustomerID             str
CUS_BillName               str
INV_TerritoryCodes         str
CUS_BillContactName        str
CUS_ShipMethod             str
dtype: object


## Observations - Sales_DataSet

- INV_InvoiceDate is a string - needs conversion to datetime 



## Get Schema Statistics

Look for duplicates and missing values

In [29]:
#get column statistics
modETL.funcGetStatistics(dfSales_DataSet)

DataFrame Statistics:
          Unnamed: 0  INV_SOLineNbr  INV_InvoiceQty  INV_InvoiceAmt
count  189151.000000  189151.000000   189151.000000   189151.000000
mean    94575.000000       3.500283       21.540247      296.874155
std     54603.334721       4.690949      129.116010     1591.709197
min         0.000000       1.000000        1.000000    -1723.440000
25%     47287.500000       1.000000        1.000000       12.500000
50%     94575.000000       2.000000        3.000000       47.320000
75%    141862.500000       4.000000       10.000000      161.210000
max    189150.000000     104.000000    10000.000000   124635.000000


DataFrame Shape:
Rows: 189151, Columns: 15


Missing Values Per Column:
INV_ProdFam          - 407: Missing Values, 0.22% Percent Missing Values
INV_TerritoryCodes   - 246: Missing Values, 0.13% Percent Missing Values
CUS_BillContactName  - 81503: Missing Values, 43.09% Percent Missing Values


Duplicate Value Count Per Column:
INV_InvoiceID        Total Values:

## Observations - Sales_DataSet

Observations:
- Missing Values in INV_ProdFam, INV_TerritoryCodes and CUS_BillContactName
- High duplicates expected in shown columns as nature of data in them would create this situation



---

## Get Unique Values - Sales_DataSet

In [30]:
#show unique values count
modETL.funcGetUniqueValuesCount(dfSales_DataSet)

DataFrame Unique Values Per Column:
Unnamed: 0           - 189151: Unique Values Out Of 189151 Total Values
INV_InvoiceID        - 78001: Unique Values Out Of 189151 Total Values
INV_InvoiceDate      - 1691: Unique Values Out Of 189151 Total Values
INV_SalesOrderID     - 66793: Unique Values Out Of 189151 Total Values
INV_SOLineNbr        - 104: Unique Values Out Of 189151 Total Values
INV_ItemID           - 7558: Unique Values Out Of 189151 Total Values
IMA_ItemName         - 5862: Unique Values Out Of 189151 Total Values
INV_ProdFam          - 130: Unique Values Out Of 188744 Total Values
INV_InvoiceQty       - 592: Unique Values Out Of 189151 Total Values
INV_InvoiceAmt       - 32082: Unique Values Out Of 189151 Total Values
INV_CustomerID       - 1928: Unique Values Out Of 189151 Total Values
CUS_BillName         - 1597: Unique Values Out Of 189151 Total Values
INV_TerritoryCodes   - 34: Unique Values Out Of 188905 Total Values
CUS_BillContactName  - 777: Unique Values Out Of 10764

## Observations - Sales_DataSet

The columns where I would expect to see high unique values are:
- INV_InvoiceID
- INV_SalesOrderID
- INV_InvoiceAmt
- *INV_InvoiceDate

This is shown in the results

* INV_InvoiceDate is not showing high unique value because it is currently a string with 00:00:00 at the end
  confident when converted to datetime this will be resolved



## Get Categorical Value Distribution

In [31]:
#get categorical distribution  
modETL.funcGetCategoricalValueDistribution(dfSales_DataSet)


DataFrame Categorical Value Distribution:
[INV_InvoiceID] Value Distribution:
INV_InvoiceID
271275    61
198715    51
214246    44
252525    40
226521    39
          ..
272068     1
272084     1
272075     1
272062     1
272092     1
Name: count, Length: 78001, dtype: int64


[INV_InvoiceDate] Value Distribution:
INV_InvoiceDate
28/04/2011    515
29/07/2011    492
30/11/2012    485
31/10/2011    449
31/07/2012    429
             ... 
01/09/2016     26
27/05/2014      9
01/04/2014      4
01/02/2013      2
03/04/2012      1
Name: count, Length: 1691, dtype: int64


[INV_SalesOrderID] Value Distribution:
INV_SalesOrderID
Misc Inv    407
155671       92
168267       72
220144       64
165933       52
           ... 
220878        1
220870        1
220409        1
220984        1
220980        1
Name: count, Length: 66793, dtype: int64


[INV_ItemID] Value Distribution:


INV_ItemID
CARRIAGE                     14752
L19EB                         4216
FPA-236                       2355
FPA-235                       2210
FPA-1009                      2096
                             ...  
ODS20/NM3/IP/220V60HZ EXP        1
AR2X8/NM3/AD2 ORB                1
CHE2X8/NM3/AD2 ORB               1
ED8/NM3/AD2 ORB                  1
DST/ODETSCREW KIT                1
Name: count, Length: 7558, dtype: int64


[IMA_ItemName] Value Distribution:
IMA_ItemName
UK Carrier Service                                                                     15674
Legend  L19  For E/B 379 X 186 X 2mm Opal Polycarbonate                                 4541
S65 Optical Detector.  Apollo Series 65 Conventional Devices                            2564
S65 Detectors Base with Diode.  Apollo Series 65 Conventional Devices                   2417
8 Watt Economy NM3 Eden with White Base & Fresnel Diffuser.                             2321
                                                 

## Observations - Get Categorical Value Distribution

Some very interesting statistics here, a lot of the categorical columns such asINV_TerritoryCodes, show effectively the most common values which in itself is an interesting piece of analysis

# Section 2

- If missing values determine what to fill with (mean, mode or categorical something else)
- Transform data types if necessary


In [32]:
## Create Copy Of DataFrame
dfSales_DataSet_Work = dfSales_DataSet.copy()

## Strip Spaces From Date Column

In [33]:
#remove spaces from Date column (string datatype)
dfSales_DataSet_Work["INV_InvoiceDate"] = dfSales_DataSet_Work["INV_InvoiceDate"].str.strip()
          
#show first 50 rows to check    
dfSales_DataSet_Work.head(50)    

,Unnamed: 0,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod
0,0,180851,01/03/2011,140079,3,17-972,Metalwork-Galvanised Retaining Brackets For Ca...,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
1,1,180851,01/03/2011,140079,4,17-625,M4 x 40mm Steel Pozi Pan Head M/Screw BZP Plated,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
2,2,180851,01/03/2011,140079,2,17-795/WHT,Metalwork White Recessing Frame Bewdley,Accessories,1.0,6.21,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
3,3,180834,01/03/2011,141347,1,SAV8/24/D/A ORB,8W 24V Ac/Dc Savona Edge Lit E/Sign Head Brass...,Savona,1.0,76.14,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
4,4,180834,01/03/2011,141347,2,l1919el-a,Legend Printed L19 Both Sides Suitable For All...,Luminaire Component,1.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
5,5,180860,01/03/2011,141420,1,gn8/m3/ex orb,Geneva 8W M3 Edge-Lit Exit. White Finish,Geneva,6.0,402.72,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
6,6,180860,01/03/2011,141420,2,l1620el-b,Legend Printed L16 One Side L20 Other Suitable...,Luminaire Component,4.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
7,7,180847,01/03/2011,141483,1,LV28C/BR ORB,INACTIVE Lavaine 28W 2D Mains Brass Style C,Lavaine,1.0,48.69,S1364,City Electrical Factors (Hinckley Group),North West,Richard Whitehurst,UK Carrier
8,8,180860,01/03/2011,141420,3,l19el-b,Legend Printed L19 S/Sided Suitable For All Ed...,Accessories,2.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
9,9,180832,01/03/2011,141642,3,TLC/3M TLC,8 Watt Economy M3 Eden with White Base & Clear...,Eden,19.0,228.95,S1736,TLC (SOUTHERN) Limited,South Coast,Roger Meakin,UK Carrier


## Observations - Sales_DataSet

No data corruptions

## Validate Numerical Values For Columns Used In Analysis

- see if INV_InvoiceQty has any values less than 1 
- see if INV_InvoiceAmt has any values less than 1


In [34]:
#get count of how many values in INV_InvoiceQty column are less than 1
intLessThanOne = (dfSales_DataSet_Work["INV_InvoiceQty"] < 1).sum()
#print result
print(f"Number of Values In INV_InvoiceQty Less Than 1: {intLessThanOne}")

#get count of how many values in INV_InvoiceAmt column are less than 1
intLessThanOne = (dfSales_DataSet_Work["INV_InvoiceAmt"] < 1).sum()
#print result
print(f"Number of Values In INV_InvoiceAmt Less Than 1: {intLessThanOne}")



Number of Values In INV_InvoiceQty Less Than 1: 0
Number of Values In INV_InvoiceAmt Less Than 1: 32766


---

## Observations - Sales_DataSet

With INV_InvoiceAmt having some values less than 1, is unusual, could be a credit note, due to the volume
will treat as such for the time being 

## Data Transformations/Feature Engineering

Convert INV_InvoiceDate column to datetime format

In [35]:
#convert Date column to datetime format
dfSales_DataSet_Work["INV_InvoiceDate"] = pd.to_datetime(dfSales_DataSet["INV_InvoiceDate"], format="%d/%m/%Y")

#check results
dfSales_DataSet_Work.head(50)

,Unnamed: 0,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod
0,0,180851,2011-03-01,140079,3,17-972,Metalwork-Galvanised Retaining Brackets For Ca...,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
1,1,180851,2011-03-01,140079,4,17-625,M4 x 40mm Steel Pozi Pan Head M/Screw BZP Plated,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
2,2,180851,2011-03-01,140079,2,17-795/WHT,Metalwork White Recessing Frame Bewdley,Accessories,1.0,6.21,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
3,3,180834,2011-03-01,141347,1,SAV8/24/D/A ORB,8W 24V Ac/Dc Savona Edge Lit E/Sign Head Brass...,Savona,1.0,76.14,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
4,4,180834,2011-03-01,141347,2,l1919el-a,Legend Printed L19 Both Sides Suitable For All...,Luminaire Component,1.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
5,5,180860,2011-03-01,141420,1,gn8/m3/ex orb,Geneva 8W M3 Edge-Lit Exit. White Finish,Geneva,6.0,402.72,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
6,6,180860,2011-03-01,141420,2,l1620el-b,Legend Printed L16 One Side L20 Other Suitable...,Luminaire Component,4.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
7,7,180847,2011-03-01,141483,1,LV28C/BR ORB,INACTIVE Lavaine 28W 2D Mains Brass Style C,Lavaine,1.0,48.69,S1364,City Electrical Factors (Hinckley Group),North West,Richard Whitehurst,UK Carrier
8,8,180860,2011-03-01,141420,3,l19el-b,Legend Printed L19 S/Sided Suitable For All Ed...,Accessories,2.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
9,9,180832,2011-03-01,141642,3,TLC/3M TLC,8 Watt Economy M3 Eden with White Base & Clear...,Eden,19.0,228.95,S1736,TLC (SOUTHERN) Limited,South Coast,Roger Meakin,UK Carrier


## Fix Missing INV_ProdFam Values

See if we can fill in the missing values in INV_ProdFam column based on the IMA_ItemID column
Any we cannot fill we will delete 

In [36]:
#find how many INV_ProdFam values are missing
dfTemp = dfSales_DataSet_Work[dfSales_DataSet_Work["INV_ProdFam"].isnull()]
#show results
dfTemp.head(100)


,Unnamed: 0,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod
1285,1285,181548,2011-03-16,Misc Inv,2,L19EL-A,Legend Printed L19 S/Sided Suitable For All Ed...,NaN,2.0,0.00,S1534,Edmundson Electrical,Scotland,NaN,UK Carrier
1387,1387,181546,2011-03-16,Misc Inv,3,L19EL-A,Legend Printed L19 S/Sided Suitable For All Ed...,NaN,1.0,0.00,S639,City Electrical Factors (Glasgow South Group),Scotland,Robert Shields,UK Carrier
1388,1388,181555,2011-03-16,Misc Inv,1,PIC8/M3/E,INACTIVE Pico Maintained 8W Economy Version (S...,NaN,2.0,66.40,S1955,Holland House Electrical Ltd,Scotland,NaN,Amtrak
1389,1389,181545,2011-03-16,Misc Inv,2,L19EL-A,Legend Printed L19 S/Sided Suitable For All Ed...,NaN,1.0,0.00,S639,City Electrical Factors (Glasgow South Group),Scotland,Robert Shields,UK Carrier
1390,1390,181547,2011-03-16,Misc Inv,2,L19EL-A,Legend Printed L19 S/Sided Suitable For All Ed...,NaN,4.0,0.00,S639,City Electrical Factors (Glasgow South Group),Scotland,Robert Shields,UK Carrier
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16949,16949,189968,2011-09-08,Misc Inv,2,L19EB,Legend L19 For E/B 379 X 186 X 2mm Opal Poly...,NaN,1.0,0.00,S1027,Allied Electrical Wholesale Limited,Scotland,Patrick Reilly,UK Carrier
17134,17134,190085,2011-09-12,Misc Inv,5,FP-019,Use FP-314 Conventional Panel with 8 Zones f...,NaN,1.0,269.00,S1579,Sabel Bennett & Fountain (Edmundson 520),West Midlands (East),NaN,UK Carrier
17135,17135,190085,2011-09-12,Misc Inv,4,FP-018,Use FP-313 Conventional Panel with 4 Zones fo...,NaN,1.0,160.00,S1579,Sabel Bennett & Fountain (Edmundson 520),West Midlands (East),NaN,UK Carrier
17136,17136,190085,2011-09-12,Misc Inv,3,FPA-008,Red Electronic sounder103 dBA @ 1Mtr 12mA,NaN,7.0,70.00,S1579,Sabel Bennett & Fountain (Edmundson 520),West Midlands (East),NaN,UK Carrier


In [37]:
#now update INV_ProdFam based on its related column: INV_ItemID
modETL.funcUpdateRelatedRecords(dfSales_DataSet_Work, "INV_ItemID", "INV_ProdFam")
#show results
dfSales_DataSet_Work.head(100)

,Unnamed: 0,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod
0,0,180851,2011-03-01,140079,3,17-972,Metalwork-Galvanised Retaining Brackets For Ca...,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
1,1,180851,2011-03-01,140079,4,17-625,M4 x 40mm Steel Pozi Pan Head M/Screw BZP Plated,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
2,2,180851,2011-03-01,140079,2,17-795/WHT,Metalwork White Recessing Frame Bewdley,Accessories,1.0,6.21,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
3,3,180834,2011-03-01,141347,1,SAV8/24/D/A ORB,8W 24V Ac/Dc Savona Edge Lit E/Sign Head Brass...,Savona,1.0,76.14,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
4,4,180834,2011-03-01,141347,2,l1919el-a,Legend Printed L19 Both Sides Suitable For All...,Luminaire Component,1.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,180889,2011-03-02,141716,3,FPA-547,Mains Voltage Safety Isolator Switch,Fire Product,1.0,21.19,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier
96,96,180889,2011-03-02,141716,6,FPA-266,XP95 Optical Smoke Detector. Apollo XP95 Dev...,Fire Product,17.0,365.36,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier
97,97,180889,2011-03-02,141716,9,FPA-277,XP95 24V Input/Output Unit with Isolator & Bac...,Fire Product,2.0,84.01,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier
98,98,180889,2011-03-02,141716,10,FPA-263,XP95 55°C Heat Detector. Apollo XP95 Devices,Fire Product,2.0,42.98,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier


## Observations - Fix Missing INV_ProdFam Values

Function worked! INV_ProdFam has no missing values

## Fix Missing INV_TerritoryCodes Values

See if we can fill in the missing values in INV_TerritoryCodes column based on the IMA_CustomerID column
Any we cannot fill we will delete 

In [38]:
#now update INV_TerritoryCodes based on its related column: INV_CustomerID
modETL.funcUpdateRelatedRecords(dfSales_DataSet_Work, "INV_CustomerID", "INV_TerritoryCodes")
#show results
dfSales_DataSet_Work.head(100)

,Unnamed: 0,INV_InvoiceID,INV_InvoiceDate,INV_SalesOrderID,INV_SOLineNbr,INV_ItemID,IMA_ItemName,INV_ProdFam,INV_InvoiceQty,INV_InvoiceAmt,INV_CustomerID,CUS_BillName,INV_TerritoryCodes,CUS_BillContactName,CUS_ShipMethod
0,0,180851,2011-03-01,140079,3,17-972,Metalwork-Galvanised Retaining Brackets For Ca...,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
1,1,180851,2011-03-01,140079,4,17-625,M4 x 40mm Steel Pozi Pan Head M/Screw BZP Plated,Luminaire Component,2.0,0.00,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
2,2,180851,2011-03-01,140079,2,17-795/WHT,Metalwork White Recessing Frame Bewdley,Accessories,1.0,6.21,S2128,Edmundson Electrical Ltd,South West,NaN,UK Carrier
3,3,180834,2011-03-01,141347,1,SAV8/24/D/A ORB,8W 24V Ac/Dc Savona Edge Lit E/Sign Head Brass...,Savona,1.0,76.14,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
4,4,180834,2011-03-01,141347,2,l1919el-a,Legend Printed L19 Both Sides Suitable For All...,Luminaire Component,1.0,0.00,S2061,Western (ACCOUNT CLOSED) Electrical Limited,South West,NaN,Despatch Next Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,180889,2011-03-02,141716,3,FPA-547,Mains Voltage Safety Isolator Switch,Fire Product,1.0,21.19,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier
96,96,180889,2011-03-02,141716,6,FPA-266,XP95 Optical Smoke Detector. Apollo XP95 Dev...,Fire Product,17.0,365.36,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier
97,97,180889,2011-03-02,141716,9,FPA-277,XP95 24V Input/Output Unit with Isolator & Bac...,Fire Product,2.0,84.01,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier
98,98,180889,2011-03-02,141716,10,FPA-263,XP95 55°C Heat Detector. Apollo XP95 Devices,Fire Product,2.0,42.98,S1539,Edmundson-Harding (268),West Midlands (East),NaN,UK Carrier


### Observations - Fix Missing INV_TerritoryCodes Values

Function worked! INV_TerritoryCodes has no missing values

## Remove Unnecessary Columns

Will remove:
- Unnamed column as it is an index column
- INV_SalesOrderID as it is not required for analysis
- INV_SOLineNbr as it is not required for analysis
- CUS_BillName as it is not required for analysis also might contravien GDPR as it contains a business name
  and this data will be on a public server so will remove it to be safe
- CUS_BillContactlName as it is not required for analysis also might contravien GDPR as it contains a business name
  and this data will be on a public server so will remove it to be safe


In [39]:
#going to remove:
dfSales_DataSet_Work.drop(columns=["Unnamed: 0","INV_SalesOrderID", "INV_SOLineNbr", "CUS_BillName", 
                                   "CUS_BillContactName"], inplace=True)

#check results
dfSales_DataSet_Work.info()

<class 'pandas.DataFrame'>
RangeIndex: 189151 entries, 0 to 189150
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   INV_InvoiceID       189151 non-null  str           
 1   INV_InvoiceDate     189151 non-null  datetime64[us]
 2   INV_ItemID          189151 non-null  str           
 3   IMA_ItemName        189151 non-null  str           
 4   INV_ProdFam         189151 non-null  str           
 5   INV_InvoiceQty      189151 non-null  float64       
 6   INV_InvoiceAmt      189151 non-null  float64       
 7   INV_CustomerID      189151 non-null  str           
 8   INV_TerritoryCodes  189151 non-null  str           
 9   CUS_ShipMethod      189151 non-null  str           
dtypes: datetime64[us](1), float64(2), str(7)
memory usage: 33.2 MB


## Rename Columns

Will rename *all* columns being used 

In [40]:
#rename rest of the columns

dfSales_DataSet_Work.rename(columns={
    "INV_ItemID": "ItemID", 
    "INV_ProdFam": "ProductFamily",
    "IMA_ItemName": "ItemName",
    "INV_ItemID": "ItemID",
    "INV_InvoiceID": "InvoiceID",
    "INV_InvoiceDate": "InvoiceDate",
    "INV_InvoiceQty": "InvoiceQty",
    "INV_InvoiceAmt": "InvoiceAmt",
    "INV_CustomerID": "CustomerID",
    "INV_TerritoryCodes": "TerritoryCodes",
    "CUS_BillContactName": "ContactName",
    "CUS_ShipMethod": "ShipMethod",
    "CUS_BillName": "CustomerName"
    }, inplace=True)

#show results
dfSales_DataSet_Work.info()

<class 'pandas.DataFrame'>
RangeIndex: 189151 entries, 0 to 189150
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   InvoiceID       189151 non-null  str           
 1   InvoiceDate     189151 non-null  datetime64[us]
 2   ItemID          189151 non-null  str           
 3   ItemName        189151 non-null  str           
 4   ProductFamily   189151 non-null  str           
 5   InvoiceQty      189151 non-null  float64       
 6   InvoiceAmt      189151 non-null  float64       
 7   CustomerID      189151 non-null  str           
 8   TerritoryCodes  189151 non-null  str           
 9   ShipMethod      189151 non-null  str           
dtypes: datetime64[us](1), float64(2), str(7)
memory usage: 33.2 MB


## Observations - Sales_Features_DataSet

Columns confirmed removed:

INV_SalesOrderID, INV_SOLineNbr

All other columns renamed

## Feature Engineering

Need to add columns:
- day
- month
- year
- quarter


In [41]:
#add day, month and year columns and a yearmonth column
dfSales_DataSet_Work["Day"] = dfSales_DataSet_Work["InvoiceDate"].dt.day
dfSales_DataSet_Work["Month"] = dfSales_DataSet_Work["InvoiceDate"].dt.month    
dfSales_DataSet_Work["Year"] = dfSales_DataSet_Work["InvoiceDate"].dt.year
dfSales_DataSet_Work["Quarter"] = dfSales_DataSet_Work["InvoiceDate"].dt.quarter
dfSales_DataSet_Work["YearMonth"] = (dfSales_DataSet_Work["InvoiceDate"].dt.year.astype(str) + 
                                     dfSales_DataSet_Work["InvoiceDate"].dt.month.astype(str))
#convert to int
dfSales_DataSet_Work["YearMonth"] =dfSales_DataSet_Work["YearMonth"].astype(int)

#show results
dfSales_DataSet_Work.head(5000)

,InvoiceID,InvoiceDate,ItemID,ItemName,ProductFamily,InvoiceQty,InvoiceAmt,CustomerID,TerritoryCodes,ShipMethod,Day,Month,Year,Quarter,YearMonth
0,180851,2011-03-01,17-972,Metalwork-Galvanised Retaining Brackets For Ca...,Luminaire Component,2.0,0.00,S2128,South West,UK Carrier,1,3,2011,1,20113
1,180851,2011-03-01,17-625,M4 x 40mm Steel Pozi Pan Head M/Screw BZP Plated,Luminaire Component,2.0,0.00,S2128,South West,UK Carrier,1,3,2011,1,20113
2,180851,2011-03-01,17-795/WHT,Metalwork White Recessing Frame Bewdley,Accessories,1.0,6.21,S2128,South West,UK Carrier,1,3,2011,1,20113
3,180834,2011-03-01,SAV8/24/D/A ORB,8W 24V Ac/Dc Savona Edge Lit E/Sign Head Brass...,Savona,1.0,76.14,S2061,South West,Despatch Next Day,1,3,2011,1,20113
4,180834,2011-03-01,l1919el-a,Legend Printed L19 Both Sides Suitable For All...,Luminaire Component,1.0,0.00,S2061,South West,Despatch Next Day,1,3,2011,1,20113
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,183671,2011-04-28,FPA-127,220V Mains Heat Detector Tester,Fire Product,1.0,170.60,S1983,South Coast,UK Carrier,28,4,2011,2,20114
4996,183791,2011-04-28,16-502,Battery Lead Acid 6V 12Ah,Battery Component,60.0,0.00,121,Non EU,Export,28,4,2011,2,20114
4997,184100,2011-04-28,SAV8/NM3/WH/R ORB,8W NM3 Savona Recessed Edge Lit E/Sign Head White,Savona,1.0,40.01,628,Non EU,Export,28,4,2011,2,20114
4998,184100,2011-04-28,L1620EL-A,Legend Printed L16 One Side L20 Other Suitable...,Luminaire Component,1.0,0.00,628,Non EU,Export,28,4,2011,2,20114


## Observations - Feature Engineering



## Save Work DataFrame To CSV File

In [42]:
#create new DataFrame for cleaned data
dfCleaned = dfSales_DataSet_Work.copy()

#save the cleaned DataFrame to a new file
modETL.funcSaveDataFrameToCleanedFile(dfCleaned)



Saved: Sales_InvoiceData To Visualisation Folder


# Conclusions and Next Steps

ETL went well, DataSet ready for use